# Clase 6 · E07: joins, cardinalidad y ventanas
Sábado 17 de octubre de 2026. Ejecución propia y entrega dentro de clase. Preparado por el PhD Esteban Hernández, CyberColombia.

[Taller E07](../../talleres/E07.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

**Mañana, antes del receso:** reejecución de E07, lectura de productos, claves y joins.
**Mañana, después del receso:** planes, broadcast, ventanas, ranking y equivalencia.
**Tarde, antes del receso:** [E08 Geometrías e intersección](E08_Geoespacial.ipynb).
**Tarde, después del receso:** [E09 Suelo y profundidad](E09_Geoespacial.ipynb), revisión del soporte espacial y entrega integrada.

## Prerrequisitos
Completar [05 Spark](05_Spark_lectura_y_agregacion.ipynb), que prepara EVA y DIVIPOLA desde los CSV. Usar el mismo `.venv`, JDK 21 y kernel. Este notebook lee sus productos explícitamente y no ejecuta el anterior. Las salidas siguen en `kit/salidas/E07`. Cerrar la sesión del notebook 05 antes de iniciar esta.

In [12]:
from pathlib import Path
import os, sys, json
import pandas as pd
import duckdb
from IPython.display import display
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
# Linux/macOS: repositorio = Path.home() / "bigdata/big-data-postgraduate"
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas/E07"
assert KIT.is_dir(), "Ajusta repositorio a tu clon local"
OUT.mkdir(parents=True, exist_ok=True)
# Los workers usan el mismo Python del kernel y su .venv.
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("E07-aprendizaje")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC").getOrCreate())
print("Python:", sys.executable, "Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

Python: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python Spark: 4.0.1
Master: local[2]
Shuffle: 4


In [13]:
for nombre in ["dim_municipio.parquet", "rendimiento_duckdb.parquet", "spark_rendimiento"]:
    assert (OUT / nombre).exists(), "Completa el notebook 05: falta " + nombre
rendimiento = spark.read.parquet(str(OUT / "spark_rendimiento"))
municipios = spark.read.parquet(str(OUT / "dim_municipio.parquet"))
claves = ["codigo_municipio", "cultivo", "estado_fisico", "anio"]
rendimiento.printSchema()
municipios.show(5, truncate=False)

root
 |-- codigo_municipio: string (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- estado_fisico: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- produccion_t: double (nullable = true)
 |-- area_ha: double (nullable = true)
 |-- rendimiento_t_ha: double (nullable = true)

+----------------+-------------------+----------+------------+
|codigo_municipio|codigo_departamento|municipio |departamento|
+----------------+-------------------+----------+------------+
|05001           |05                 |MEDELLÍN  |ANTIOQUIA   |
|05002           |05                 |ABEJORRAL |ANTIOQUIA   |
|05004           |05                 |ABRIAQUÍ  |ANTIOQUIA   |
|05021           |05                 |ALEJANDRÍA|ANTIOQUIA   |
|05030           |05                 |AMAGÁ     |ANTIOQUIA   |
+----------------+-------------------+----------+------------+
only showing top 5 rows


## 1. Comprobar la dimensión antes del join
Una dimensión duplicada multiplicaría filas del resultado. Spark no ofrece `validate="many_to_one"` en `join`; construir explícitamente las comprobaciones. Un código no encontrado se reporta, no se elimina.

In [14]:
duplicados = municipios.groupBy("codigo_municipio").count().filter(F.col("count") > 1)
assert duplicados.count() == 0, "Revisar duplicados de la dimensión antes de unir"
assert municipios.filter(F.col("codigo_municipio").isNull()).count() == 0
sin_dimension = rendimiento.join(municipios.select("codigo_municipio"), "codigo_municipio", "left_anti")
sin_dimension.select("codigo_municipio").distinct().show(truncate=False)
sin_dimension.write.mode("overwrite").parquet(str(OUT / "revision_sin_dimension"))
filas_antes = rendimiento.count()
print("Grupos antes:", filas_antes, "sin dimensión:", sin_dimension.count())

+----------------+
|codigo_municipio|
+----------------+
+----------------+



Grupos antes: 107620 sin dimensión: 0


## 2. LEFT JOIN normal y broadcast explícito
Comparar planes y conservar el número de grupos. Spark puede elegir broadcast automáticamente con una dimensión pequeña; observar el plan real en lugar de presuponer una estrategia. El hint es razonable solo si la dimensión cabe en cada executor; aquí se trabaja en una única máquina.

In [15]:
union_normal = rendimiento.join(municipios, "codigo_municipio", "left")
union_broadcast = rendimiento.join(F.broadcast(municipios), "codigo_municipio", "left")
union_normal.explain("formatted")
union_broadcast.explain("formatted")
assert union_normal.count() == filas_antes
assert union_broadcast.count() == filas_antes

== Physical Plan ==
AdaptiveSparkPlan (7)
+- Project (6)
   +- BroadcastHashJoin LeftOuter BuildRight (5)
      :- Scan parquet  (1)
      +- BroadcastExchange (4)
         +- Filter (3)
            +- Scan parquet  (2)


(1) Scan parquet 
Output [7]: [codigo_municipio#187, cultivo#188, estado_fisico#189, anio#190L, produccion_t#191, area_ha#192, rendimiento_t_ha#193]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/spark_rendimiento]
ReadSchema: struct<codigo_municipio:string,cultivo:string,estado_fisico:string,anio:bigint,produccion_t:double,area_ha:double,rendimiento_t_ha:double>

(2) Scan parquet 
Output [4]: [codigo_municipio#194, codigo_departamento#195, municipio#196, departamento#197]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/dim_municipio.parquet]
PushedFilters: [IsNotNull(codigo_municipio)]
ReadSchema: struct<codigo_municipio:string,codigo_dep

### Medir la materialización del join
Incluye lectura y escritura; el plan se construyó antes. `count()` por sí solo no representa el coste de producir todas las columnas. No se compara rendimiento entre estrategias con una sola vuelta.

In [16]:
%%time
union_broadcast.write.mode("overwrite").parquet(str(OUT / "rendimiento_municipal"))

CPU times: user 5.75 ms, sys: 1.36 ms, total: 7.11 ms
Wall time: 4.3 s


In [17]:
normal_pd = union_normal.toPandas().sort_values(claves).reset_index(drop=True)
union_pd = pd.read_parquet(OUT / "rendimiento_municipal").sort_values(claves).reset_index(drop=True)
pd.testing.assert_frame_equal(normal_pd[union_pd.columns], union_pd, check_dtype=False)
display(union_pd.head())
# Solo el resultado agregado pequeño se lleva a pandas para verificar.

,codigo_municipio,cultivo,estado_fisico,anio,produccion_t,area_ha,rendimiento_t_ha,codigo_departamento,municipio,departamento
0,05001,Aguacate,En fresco,2019,138.00,23.00,6.000000,05,MEDELLÍN,ANTIOQUIA
1,05001,Aguacate,En fresco,2020,114.00,19.00,6.000000,05,MEDELLÍN,ANTIOQUIA
2,05001,Aguacate,En fresco,2021,117.12,19.52,6.000000,05,MEDELLÍN,ANTIOQUIA
3,05001,Aguacate,En fresco,2022,141.12,23.52,6.000000,05,MEDELLÍN,ANTIOQUIA
4,05001,Aguacate,En fresco,2023,128.12,19.52,6.563525,05,MEDELLÍN,ANTIOQUIA


## 3. Ventana: tres municipios por cultivo, estado y año
`row_number` asigna posiciones sin colapsar filas. Particionar por cultivo, estado y año, ordenar rendimiento descendente y desempatar por código municipal ascendente. El desempate permite reproducir el top 3; no implica superioridad agronómica ni causalidad.

In [18]:
from pyspark.sql.window import Window
ventana = Window.partitionBy("cultivo", "estado_fisico", "anio").orderBy(
    F.col("rendimiento_t_ha").desc(), F.col("codigo_municipio").asc())
ranking = rendimiento.withColumn("posicion", F.row_number().over(ventana))
top3 = ranking.filter(F.col("posicion") <= 3)
top3.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Filter (8)
   +- Window (7)
      +- WindowGroupLimit (6)
         +- Sort (5)
            +- Exchange (4)
               +- WindowGroupLimit (3)
                  +- Sort (2)
                     +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#187, cultivo#188, estado_fisico#189, anio#190L, produccion_t#191, area_ha#192, rendimiento_t_ha#193]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/spark_rendimiento]
ReadSchema: struct<codigo_municipio:string,cultivo:string,estado_fisico:string,anio:bigint,produccion_t:double,area_ha:double,rendimiento_t_ha:double>

(2) Sort
Input [7]: [codigo_municipio#187, cultivo#188, estado_fisico#189, anio#190L, produccion_t#191, area_ha#192, rendimiento_t_ha#193]
Arguments: [cultivo#188 ASC NULLS FIRST, estado_fisico#189 ASC NULLS FIRST, anio#190L ASC NULLS FIRST, rendimiento_t_ha#193 DESC NULLS LAST, codigo_municipio#187 A

In [19]:
%%time
top3.write.mode("overwrite").parquet(str(OUT / "spark_top3"))

CPU times: user 5.92 ms, sys: 369 μs, total: 6.29 ms
Wall time: 4.33 s


## 4. Verificar el ranking con SQL en DuckDB
Para aislar la semántica de la ventana, ambos motores leen el mismo resultado agregado Spark. La equivalencia numérica del agregado contra DuckDB ya se verificó en 05. Evitamos que diferencias de coma flotante mínimas alteren un empate aparente.

In [20]:
con = duckdb.connect()
con.read_parquet(str(OUT / "spark_rendimiento/*.parquet")).create_view("rendimiento")
sql_top3 = """
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY cultivo, estado_fisico, anio
    ORDER BY rendimiento_t_ha DESC, codigo_municipio ASC
  ) AS posicion
  FROM rendimiento
) WHERE posicion <= 3
"""
top_duckdb = con.sql(sql_top3).df()
top_spark = pd.read_parquet(OUT / "spark_top3")
orden = ["cultivo", "estado_fisico", "anio", "posicion"]
pd.testing.assert_frame_equal(
    top_duckdb.sort_values(orden).reset_index(drop=True)[top_spark.columns],
    top_spark.sort_values(orden).reset_index(drop=True), check_dtype=False)
assert top_spark.groupby(["cultivo", "estado_fisico", "anio"]).size().le(3).all()
top_spark.sort_values(orden).to_csv(OUT / "top3_verificado.csv", index=False)
display(top_spark.sort_values(orden).head(12))
con.close()

,codigo_municipio,cultivo,estado_fisico,anio,produccion_t,area_ha,rendimiento_t_ha,posicion
0,05440,Acelga,En fresco,2019,185.50,7.00,26.500000,1
1,25175,Acelga,En fresco,2019,500.00,20.00,25.000000,2
2,25473,Acelga,En fresco,2019,714.00,34.00,21.000000,3
3,05440,Acelga,En fresco,2020,265.00,10.00,26.500000,1
4,25473,Acelga,En fresco,2020,693.00,33.00,21.000000,2
5,25214,Acelga,En fresco,2020,253.20,12.66,20.000000,3
6,05440,Acelga,En fresco,2021,238.50,9.00,26.500000,1
7,25473,Acelga,En fresco,2021,1560.00,70.00,22.285714,2
8,68147,Acelga,En fresco,2021,60.00,3.00,20.000000,3
9,05440,Acelga,En fresco,2022,198.75,7.50,26.500000,1


## 5. Cambiar particiones conservando el significado
Comparar 4 y 2 particiones de shuffle sin cambiar la consulta. Reconstruir el plan después de cambiar la configuración. La ejecución adaptativa puede combinar particiones, por lo que el número de archivos no tiene que ser exactamente 4 o 2. No confundir particiones con máquinas.

In [21]:
eva = spark.read.parquet(str(OUT / "eva.parquet"))
evidencia = []
for numero in [4, 2]:
    spark.conf.set("spark.sql.shuffle.partitions", str(numero))
    consulta = (eva.filter(F.col("apto_rendimiento")).groupBy(*claves)
        .agg(F.sum("produccion_t").alias("produccion_t"), F.sum("area_ha").alias("area_ha"))
        .withColumn("rendimiento_t_ha", F.col("produccion_t") / F.col("area_ha")))
    consulta.explain("formatted")
    destino = OUT / f"shuffle_{numero}"
    consulta.write.mode("overwrite").parquet(str(destino))
    recuperado = pd.read_parquet(destino).sort_values(claves).reset_index(drop=True)
    original = pd.read_parquet(OUT / "spark_rendimiento").sort_values(claves).reset_index(drop=True)
    pd.testing.assert_frame_equal(original[recuperado.columns], recuperado, check_dtype=False, atol=1e-6, rtol=0)
    evidencia.append({"shuffle_configurado": numero, "archivos": len(list(destino.glob("part-*.parquet"))),
                      "grupos": len(recuperado), "equivalente": True,
                      "adaptive_enabled": spark.conf.get("spark.sql.adaptive.enabled")})
spark.conf.set("spark.sql.shuffle.partitions", "4")
display(pd.DataFrame(evidencia))

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#291, anio#293L, cultivo#294, estado_fisico#296, area_ha#297, produccion_t#298, apto_rendimiento#305]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) Filter
Input [7]: [codigo_municipio#291, anio#293L, cultivo#294, estado_fisico#296, area_ha#297, produccion_t#298, apto_rendimiento#305]
Condition : (isnotnull(apto_rendimiento#305) AND apto_rendimiento#305)

(3) Project
Output [6]: [codigo_municipio#291, anio#

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- HashAggregate (6)
      +- Exchange (5)
         +- HashAggregate (4)
            +- Project (3)
               +- Filter (2)
                  +- Scan parquet  (1)


(1) Scan parquet 
Output [7]: [codigo_municipio#291, anio#293L, cultivo#294, estado_fisico#296, area_ha#297, produccion_t#298, apto_rendimiento#305]
Batched: true
Location: InMemoryFileIndex [file:/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/E07/eva.parquet]
PushedFilters: [IsNotNull(apto_rendimiento), EqualTo(apto_rendimiento,true)]
ReadSchema: struct<codigo_municipio:string,anio:bigint,cultivo:string,estado_fisico:string,area_ha:double,produccion_t:double,apto_rendimiento:boolean>

(2) Filter
Input [7]: [codigo_municipio#291, anio#293L, cultivo#294, estado_fisico#296, area_ha#297, produccion_t#298, apto_rendimiento#305]
Condition : (isnotnull(apto_rendimiento#305) AND apto_rendimiento#305)

(3) Project
Output [6]: [codigo_municipio#291, anio#

,shuffle_configurado,archivos,grupos,equivalente,adaptive_enabled
0,4,2,107620,True,true
1,2,2,107620,True,true


## 6. Escribir el control y entregar
Explicar las comprobaciones de cardinalidad, el plan del join, el desempate y la diferencia entre configuración y archivos reales. Adjuntar el notebook ejecutado y los controles junto con los mapas E08/E09. La ventana calcula un ranking descriptivo, no una recomendación de cultivo.

In [22]:
control = {"grupos_antes": filas_antes, "grupos_despues": len(union_pd),
    "grupos_sin_dimension": sin_dimension.count(), "filas_top3": len(top_spark),
    "ranking_equivalente": True, "particiones": evidencia}
with open(OUT / "control_joins_ventanas.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
display(control)
spark.stop()

{'grupos_antes': 107620,
 'grupos_despues': 107620,
 'grupos_sin_dimension': 0,
 'filas_top3': 2990,
 'ranking_equivalente': True,
 'particiones': [{'shuffle_configurado': 4,
   'archivos': 2,
   'grupos': 107620,
   'equivalente': True,
   'adaptive_enabled': 'true'},
  {'shuffle_configurado': 2,
   'archivos': 2,
   'grupos': 107620,
   'equivalente': True,
   'adaptive_enabled': 'true'}]}